# PCA

## Import data

In [1]:
import pandas as pd

df = pd.read_csv('../../data/base_v3.csv')
df.head()

,ISRC,Track,Album Name,Artist,Release Date,All Time Rank,Track Score,Spotify Streams,Spotify Popularity,YouTube Views,...,Pandora Streams,Pandora Track Stations,Shazam Counts,Explicit Track,Length,Releases,Genres,Registration Country,Playlist Probability,High Playlist Probability
0,QM24S2402528,MILLION DOLLAR BABY,Million Dollar Baby - Single,Tommy Richman,4/26/2024,1,725.4,390470936,92,84274754,...,18004655,22931,2669262,0,155.1510,1,[],United States,0.734589,1
1,USUG12400910,Not Like Us,Not Like Us,Kendrick Lamar,5/4/2024,2,545.9,323703884,92,116347040,...,7780028,28444,1118279,1,274.1920,5,"['hip hop', 'producer tag - dj mustard']",United States,0.721077,1
2,QZJ842400387,i like the way you kiss me,I like the way you kiss me,Artemas,3/19/2024,3,538.4,601309283,92,122599116,...,5022621,5639,5285340,0,143.1865,8,['synth-pop'],United States,0.771418,1
3,USSM12209777,Flowers,Flowers - Single,Miley Cyrus,1/12/2023,4,444.9,2031280633,85,1096100899,...,190260277,203384,11822942,0,200.4530,30,"['yacht rock', 'alternative pop', 'pop rock', ...",United States,0.799485,1
4,USUG12403398,Houdini,Houdini,Eminem,5/31/2024,5,423.3,107034922,88,77373957,...,4493884,7006,457017,1,227.0320,20,[],United States,0.721544,1


## Remove imputed values for more representative clusters

In [2]:
basev1 = pd.read_csv('../../data/base_v1.csv')
basev1.head()

,ISRC,Track,Album Name,Artist,Release Date,All Time Rank,Track Score,Spotify Streams,Spotify Playlist Count,Spotify Playlist Reach,...,Pandora Streams,Pandora Track Stations,Soundcloud Streams,Shazam Counts,TIDAL Popularity,Explicit Track,Length,Releases,Genres,Registration Country
0,QM24S2402528,MILLION DOLLAR BABY,Million Dollar Baby - Single,Tommy Richman,4/26/2024,1,725.4,"390,470,936","30,716","196,631,588",...,"18,004,655","22,931","4,818,457","2,669,262",NaN,0,155.1510,1,[],United States
1,USUG12400910,Not Like Us,Not Like Us,Kendrick Lamar,5/4/2024,2,545.9,"323,703,884","28,113","174,597,137",...,"7,780,028","28,444","6,623,075","1,118,279",NaN,1,274.1920,5,"['hip hop', 'producer tag - dj mustard']",United States
2,QZJ842400387,i like the way you kiss me,I like the way you kiss me,Artemas,3/19/2024,3,538.4,"601,309,283","54,331","211,607,669",...,"5,022,621","5,639","7,208,651","5,285,340",NaN,0,143.1865,8,['synth-pop'],United States
3,USSM12209777,Flowers,Flowers - Single,Miley Cyrus,1/12/2023,4,444.9,"2,031,280,633","269,802","136,569,078",...,"190,260,277","203,384",NaN,"11,822,942",NaN,0,200.4530,30,"['yacht rock', 'alternative pop', 'pop rock', ...",United States
4,USUG12403398,Houdini,Houdini,Eminem,5/31/2024,5,423.3,"107,034,922","7,223","151,469,874",...,"4,493,884","7,006","207,179","457,017",NaN,1,227.0320,20,[],United States


In [3]:
valid_isrcs = basev1.loc[
    ~(basev1['TikTok Posts'].isnull() | 
      basev1['TikTok Likes'].isnull() | 
      basev1['TikTok Views'].isnull() |
      basev1['YouTube Views'].isnull() |
      basev1['YouTube Likes'].isnull()), 
    'ISRC'
]

df_filtered = df[df['ISRC'].isin(valid_isrcs)]
df_final = df_filtered.merge(valid_isrcs.to_frame(), on='ISRC', how='inner')
print(f'Length of dataframe with imputed values: {len(df)}')
print(f'Length of dataframe without imputed values: {len(df_final)}')

Length of dataframe with imputed values: 4572
Length of dataframe without imputed values: 3311


In [4]:
df_final['All Time Rank Bin'] = pd.qcut(df_final['All Time Rank'], q=2, labels=[1, 0]).astype(int)
df_final.head(2)

,ISRC,Track,Album Name,Artist,Release Date,All Time Rank,Track Score,Spotify Streams,Spotify Popularity,YouTube Views,...,Pandora Track Stations,Shazam Counts,Explicit Track,Length,Releases,Genres,Registration Country,Playlist Probability,High Playlist Probability,All Time Rank Bin
0,QM24S2402528,MILLION DOLLAR BABY,Million Dollar Baby - Single,Tommy Richman,4/26/2024,1,725.4,390470936,92,84274754,...,22931,2669262,0,155.151,1,[],United States,0.734589,1,1
1,USUG12400910,Not Like Us,Not Like Us,Kendrick Lamar,5/4/2024,2,545.9,323703884,92,116347040,...,28444,1118279,1,274.192,5,"['hip hop', 'producer tag - dj mustard']",United States,0.721077,1,1


## Perform PCA

### Import modules

For PCA, we use scikit-learn's PCA and KernelPCA:
1. PCA: https://scikit-learn.org/stable/modules/generated/sklearn.decomposition.PCA.html
2. KernelPCA: https://scikit-learn.org/stable/modules/generated/sklearn.decomposition.KernelPCA.html

For plotting, we use Plotly's 2D & 3D scatterplots, seaborn, and Pyplot:
1. 2D: https://plotly.com/python/line-and-scatter/
2. 3D: https://plotly.com/python/3d-scatter-plots/
3. seaborn: https://seaborn.pydata.org/
4. Pyplot: https://matplotlib.org/stable/tutorials/pyplot.html

In [5]:
# install plotly for interactive graphs
import subprocess
import sys
def install(package):
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', package, '--quiet'])
install('plotly')

import numpy as np
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import plotly.express as px
import seaborn as sns
from sklearn.decomposition import PCA, KernelPCA
from sklearn.preprocessing import StandardScaler

### Apply PCA with linear and radial kernels

In [6]:
def normalize_data(df, quantitative_cols):
    """Use standard scaler to normalize quantitative data columns."""
    scaler = StandardScaler()
    return scaler.fit_transform(df[quantitative_cols])

def apply_pca(df_scaled, df, n):
    """Apply PCA to scaled data and join with categorical labels."""
    pca = PCA(n_components=n)
    df_pca = pd.DataFrame(pca.fit_transform(df_scaled), columns=[f'PC{i+1}' for i in range(n)])
    return df_pca.assign(**df[['All Time Rank', 'All Time Rank Bin', 'Track', 'Artist']])

def apply_kernel_pca(df_scaled, df, n, kernel='rbf', gamma=0.05):
    """Apply Kernel PCA to scaled data and join with categorical labels."""
    kpca = KernelPCA(n_components=n, kernel=kernel, gamma=gamma, fit_inverse_transform=True)
    df_kpca = pd.DataFrame(kpca.fit_transform(df_scaled), columns=[f'PC{i+1}' for i in range(n)])
    return df_kpca.assign(**df[['All Time Rank', 'All Time Rank Bin', 'Track', 'Artist']])

### Run above functions

In [7]:
print('Starting Data:')
display(df_final.head())

# select quantitative columns (non-streaming, social media related)
quantitative_cols = [
    'YouTube Views', 'YouTube Likes', 'TikTok Posts', 
    'TikTok Likes', 'TikTok Views'
]

# normalize selected columns
df_scaled = normalize_data(df_final, quantitative_cols)

print('PCA-Ready Data:')
display(pd.DataFrame(df_scaled).head())

# fit and transform data points onto PC vector space
pca_df_2d = apply_pca(df_scaled, df_final, 2)
pca_df_3d = apply_pca(df_scaled, df_final, 3)
kpca_df_2d = apply_kernel_pca(df_scaled, df_final, 2)
kpca_df_3d = apply_kernel_pca(df_scaled, df_final, 3)

Starting Data:


,ISRC,Track,Album Name,Artist,Release Date,All Time Rank,Track Score,Spotify Streams,Spotify Popularity,YouTube Views,...,Pandora Track Stations,Shazam Counts,Explicit Track,Length,Releases,Genres,Registration Country,Playlist Probability,High Playlist Probability,All Time Rank Bin
0,QM24S2402528,MILLION DOLLAR BABY,Million Dollar Baby - Single,Tommy Richman,4/26/2024,1,725.4,390470936,92,84274754,...,22931,2669262,0,155.1510,1,[],United States,0.734589,1,1
1,USUG12400910,Not Like Us,Not Like Us,Kendrick Lamar,5/4/2024,2,545.9,323703884,92,116347040,...,28444,1118279,1,274.1920,5,"['hip hop', 'producer tag - dj mustard']",United States,0.721077,1,1
2,QZJ842400387,i like the way you kiss me,I like the way you kiss me,Artemas,3/19/2024,3,538.4,601309283,92,122599116,...,5639,5285340,0,143.1865,8,['synth-pop'],United States,0.771418,1,1
3,USSM12209777,Flowers,Flowers - Single,Miley Cyrus,1/12/2023,4,444.9,2031280633,85,1096100899,...,203384,11822942,0,200.4530,30,"['yacht rock', 'alternative pop', 'pop rock', ...",United States,0.799485,1,1
4,USAT22311371,Lovin On Me,Lovin On Me,Jack Harlow,11/10/2023,6,410.1,670665438,83,131148091,...,50982,4517131,1,138.4110,13,"['pop', 'rap/hip hop', 'hip hop']",United States,0.758437,1,1


PCA-Ready Data:


,0,1,2,3,4
0,-0.490914,-0.331929,2.124205,0.942921,0.678484
1,-0.446978,0.032294,-0.096977,-0.144835,-0.164481
2,-0.438414,-0.226046,0.928221,0.278608,0.355514
3,0.895187,1.499169,2.744423,1.696854,2.203774
4,-0.426702,-0.397753,1.441525,0.172345,0.284702


### Plot PCA colored by All Time Rank and Binned All Time Rank

In [8]:
def plot_pca(df_pca, dims, column, title, color_scheme):
    """
    Create 2D and 3D scatterplots of the PCA transformed data points and color by All Time Rank 
    and All Time Rank Bin categories.
    """
    hover_data = ['Track', 'Artist', column]
    
    if dims == 3:
        fig = px.scatter_3d(df_pca, x='PC1', y='PC2', z='PC3', color=column, title=title,
                            color_continuous_scale=color_scheme, hover_data=hover_data)
        fig.update_layout(
            scene_camera=dict(
                eye=dict(x=1.3, y=-1.7, z=1.3),
                center=dict(x=0, y=0, z=0),
                up=dict(x=0, y=0, z=1),
            )
        )
    else:
        fig = px.scatter(df_pca, x='PC1', y='PC2', color=column, title=title,
                         color_continuous_scale=color_scheme, hover_data=hover_data)
    
    fig.update_layout(autosize=False, width=1000, height=800)
    fig.update_traces(marker=dict(size=6, opacity=0.7))
    # save plotly's as json files to display on website
    fig.write_html(f"./pca-plots/{title.replace(' ', '')}_{dims}D.html")
    fig.show()

def find_cumevr_threshold(df_scaled, threshold):
    """Find number of PCs required to have >=`threshold` variance retention."""
    pca_full = PCA().fit(df_scaled)
    cumevr = np.cumsum(pca_full.explained_variance_ratio_)
    return np.argmax(cumevr >= threshold) + 1, pca_full

def plot_variance(ax1, ax2, evr, cumevr, threshold, pc_crit):
    """Plot EVR and cumulative EVR barplots."""
    
    # barplot of explained variance ratio
    ax1.bar(range(1, len(evr) + 1), evr, color='mediumseagreen', alpha=0.7, width=0.8)
    ax1.set_title('Explained Variance Ratio', fontsize=18)
    ax1.set_ylabel('EVR')
    ax1.set_xlabel('Principal Component')
    ax1.set_xticks(range(1, len(evr) + 1))

    # barplot of cumulative explained variance ratio
    colors = ['black' if i < pc_crit else 'lightgrey' for i in range(len(cumevr))]
    ax2.bar(range(1, len(cumevr) + 1), cumevr, color=colors, width=0.8)
    ax2.axhline(y=threshold, color='mediumseagreen', linestyle=':', label='95% Threshold', lw=3)
    ax2.set_title('Cumulative Explained Variance Ratio', fontsize=18)
    ax2.set_ylabel('Cumulative EVR')
    ax2.set_xlabel('Principal Component')
    ax2.set_xticks(range(1, len(cumevr) + 1))
    ax2.legend()

def plot_pca_heatmap(ax3, pca_full, numerical_cols):
    """Plot PCA loadings, which represent the amount that each variable contributes to a PC."""
    
    # create custom colormap
    colormap_heatmap = mcolors.LinearSegmentedColormap.from_list('custom_colormap', ['black', 'white', 'mediumseagreen'])
    
    # heatmap of the loadings
    sns.heatmap(pd.DataFrame(pca_full.components_.T, index=numerical_cols, 
                             columns=[f'PC{i+1}' for i in range(len(pca_full.components_))]), 
                             cmap=colormap_heatmap, annot=True, fmt='.2f', linewidths=0.5, ax=ax3)
    ax3.set_title('PCA Component Heatmap', fontsize=18)

### Run above functions

In [ ]:
# create custom colormaps
colormap1 = [(0.0, 'mediumseagreen'), (0.15, 'limegreen'), (0.5, 'gold'), (0.75, 'orange'), (1.0, 'red')]
colormap2 = [(0.0, 'black'), (0.5, 'white'), (1.0, 'mediumseagreen')]

"""
Using the PCA function:
"""

# plot data points in the 2D and 3D PC vector spaces
for column, color_scheme in zip(['All Time Rank', 'All Time Rank Bin'], [colormap1, colormap2]):
    plot_pca(pca_df_2d, 2, column, f'PCA Colored by {column}', color_scheme)
    plot_pca(pca_df_3d, 3, column, f'PCA Colored by {column}', color_scheme)

# find number of PCs required to have >=`threshold` variance retention
threshold = 0.95
pc_crit, pca_full = find_cumevr_threshold(df_scaled, threshold)
evr, cumevr = pca_full.explained_variance_ratio_, np.cumsum(pca_full.explained_variance_ratio_)

# plot evr, cumevr, and loadings
fig, axes = plt.subplots(1, 3, figsize=(18, 5), constrained_layout=True)
plot_variance(axes[0], axes[1], evr, cumevr, threshold, pc_crit)
plot_pca_heatmap(axes[2], pca_full, quantitative_cols)
plt.show()

# print top three largest eigenvalues
top_3_eigenvalues = np.sort(pca_full.explained_variance_)[-3:]
print('Top 3 eigenvalues:')
for eigenvalue in top_3_eigenvalues[::-1]:
    print(f'{eigenvalue:.2f}')
print(f'Sum of all eigenvalues: {np.sum(pca_full.explained_variance_):.2f}')

"""
Using the KernelPCA function with a radial basis function:
"""

for column, color_scheme in zip(['All Time Rank', 'All Time Rank Bin'], [colormap1, colormap2]):
    plot_pca(kpca_df_2d, 2, column, f'RBF Kernel PCA Colored by {column}', color_scheme)
    plot_pca(kpca_df_3d, 3, column, f'RBF Kernel PCA Colored by {column}', color_scheme)
    
# save the three principal component KernelPCA dataframe for clustering
# kpca_df_3d.to_csv('../../data/kpca3d.csv', encoding='utf-8', index=False)